A QuantumFinance realizou uma parceria com algumas empresas clientes para fornecer o score de crédito de seus clientes optantes para permitir melhores condições de pagamento.
Para que o modelo seja mais simples e diferente do modelo de score principal do banco, será necessário treinar esse modelo com os dados das transações mais recentes dos clientes.
Para permitir governança interna e integração com outros sistemas, esta solução precisa incluir:
1.Template de repositório para organização dos arquivos (dataset, notebook, modelo, etc.)
2.Rastreamento dos experimentos do treinamento do modelo
3.Versionamento do modelo
4.Disponibilização de endpoint de API seguro com autenticação e throttling
5.Documentação da API

Dataset: mesmo dataset/modelo adotado na disciplina de Machine Learning.


Entregáveis:
Um (01) arquivo .ZIP contendo:

- 1 - Um esquema/template/imagem/descritivo (qualquer formato) de como ficariam organizados os arquivos e diretórios necessários para a manutenção do ciclo de vida do projeto (os scripts, arquivos de configuração, dados, etc)
- 2 - O script (.py ou .ipynb) de treinamento do modelo onde consta como foi implementado o rastreio do modelo (quais artefatos são rastreados e como), o critério de avaliação (se vai virar o novo modelo em produção), e como será a promoção para produção (mudança de versão)
- 3 - O script (.py ou .ipynb) de inferência do modelo, mostrando como será feita a leitura do modelo que está na última versão (ou seja, o último promovido pelo script de treinamento)
- 4 - Um documento (.docx,.md, ou .pdf) da API, contendo pelo menos:
  -  a) o endpoint para acesso
  -  b) todas as informações necessárias para realizar a chamada de API (como método, payload, headers, formato de autenticação, etc) e seus formatos
  -  c) formato e exemplo da(s) resposta(s) possíveis (incluindo erros)
  -  d) troubleshooting/FAQ: principais problemas (problemas mais comuns) e possíveis soluções
  -  e) como foi feito o deploy (em caso de serviço na nuvem: passo a passo e imagens) ou como ligar/utilizar o servidor local

In [62]:
%pip install flask argon2-cffi jwt jolib azureml-core==1.59.0 azureml-defaults==1.59.0 azureml-mlflow==1.59.0 azureml-telemetry==1.59.0 azureml-dataprep[pandas] pandas flask_limiter

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor as rfr
import joblib
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, ConfusionMatrixDisplay, classification_report
import sys, sklearn, joblib, uuid, os
from azureml.core import Experiment, Workspace, Dataset, Model
import json

dataset_name = 'Risco-De-Credito'

def train_models():
  
  #Trocar os códigos abaixo pelos da sua instância!

  try:
    with open("config.json") as file: data = json.load(file) 
    ( subscription_id, resource_group, workspace_name ) = data
  except:
    print('config.json file not set')
    return
  
  workspace = Workspace(subscription_id, resource_group, workspace_name)
  
  dataset_name, dataset_version = dataset_name, None
  try:
    dataset = Dataset.get_by_name(workspace, name=dataset_name)
    print(dataset)
    dataset_version = dataset.version
    dataset.download(target_path='./data', overwrite=True)
    mydf = pd.read_csv(f'./data/{dataset_name}.csv')
    display( mydf.head() )
  except:
    print("Dataset não encontrado")
    return

  targetcol = 'default'
  y = mydf[targetcol]

  independentcols = ['renda', 'idade', 'etnia', 'sexo', 'casapropria', 'outrasrendas', 'estadocivil', 'escolaridade']
  X = mydf[independentcols]

  X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=10)

  experiment = Experiment(workspace=workspace, name="Experimento_MLStudio")

  lst_n_estimators = [1, 3, 5, 8]
  lst_max_depth = [1, 4, None]
  lst_min_samples_leaf = [8, 3]
  for n_estimators in lst_n_estimators: 
    for max_depth in lst_max_depth: 
      for min_samples_leaf in lst_min_samples_leaf: 
        myrunid = str(uuid.uuid1())
        run = experiment.start_logging(run_id=myrunid,
                                      display_name=f"My Run {myrunid}",
                                      outputs="modelo",
                                      snapshot_directory="dadostreino")
        run.log("Tipo", "Regressor")

        clf = rfr(n_estimators=n_estimators, max_depth=max_depth, min_samples_leaf=min_samples_leaf)
        clf.fit(X=X_train[independentcols], y=y_train)

        clf_acuracia = clf.score(X=X_test[independentcols], y=y_test)
        #Demais logs (não serão comparados)
        run.log("acuracia", clf_acuracia)
        run.log("language", "python")
        run.log("python", sys.version)
        run.log("Versao sklearn", sklearn.__version__)
        run.log("criterion", clf.criterion)
        run.log("n_estimators", clf.n_estimators)
        run.log("min_samples_leaf", clf.min_samples_leaf)
        run.log("max_depth",  str(clf.max_depth))
        run.log("dataset_name",  dataset_name)
        run.log("dataset_version",  dataset_version)
        run.log_list("Inputs", independentcols)

        #Exportar o(s) binário(s) do modelo
        model_name = "modelo_bin.pkl"
        binaryname = "model_temp/" + model_name
        joblib.dump(value=clf, filename=binaryname)
        run.log("model_binary", binaryname)

        #Finaliza o experimento e apaga arquivos temporários.
        run.complete()
        run.wait_for_completion()

train_models()

{'subscription_id': '29e32811-9313-4d42-80fe-98720f16c8f7', 'resource_group': 'mlops', 'workspace_name': 'mlops-workspace'}


UserErrorException: UserErrorException:
	Message: You are currently logged-in to 11dbbfe2-89b8-4549-be10-cec364e59551 tenant. You don't have access to subscription_id subscription, please check if it is in this tenant. All the subscriptions that you have access to in this tenant are = 
 [SubscriptionInfo(subscription_name='Azure for Students', subscription_id='29e32811-9313-4d42-80fe-98720f16c8f7')]. 
 Please refer to aka.ms/aml-notebook-auth for different authentication mechanisms in azureml-sdk.
	InnerException None
	ErrorResponse 
{
    "error": {
        "code": "UserError",
        "message": "You are currently logged-in to 11dbbfe2-89b8-4549-be10-cec364e59551 tenant. You don't have access to subscription_id subscription, please check if it is in this tenant. All the subscriptions that you have access to in this tenant are = \n [SubscriptionInfo(subscription_name='Azure for Students', subscription_id='29e32811-9313-4d42-80fe-98720f16c8f7')]. \n Please refer to aka.ms/aml-notebook-

In [ ]:
import shutil

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor as rfr
import os
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import Ridge
import os
from azureml.core import Experiment, Workspace, Dataset, Model

def promote_models():
    try:
      with open("config.json") as file: data = json.load(file)  
      ( subscription_id, resource_group, workspace_name ) = data
    except:
      print('config.json file not set')
      return
  
    workspace = Workspace(subscription_id, resource_group, workspace_name)
    experiment = Experiment(workspace=workspace, name="Experimento_MLStudio")

    min_acuracia = -1
    melhor_run = None
    for i, r in enumerate(experiment.get_runs()):
      print(f"Avaliando execução {i} - {r.name}.")
      if r.status != 'Completed':
        continue
      #print(r, r.get_metrics())
      r_acuracia = r.get_metrics('acuracia').get('acuracia', -1)
      if not type( r_acuracia ) == float:
        print(f"Acurácia inválida {r_acuracia}. Será considerado 0.0 de acurácia.")
        r_acuracia = 0.0
      if r_acuracia > min_acuracia:
        melhor_run = r
        min_acuracia = r_acuracia
        print(f"Melhor modelo detectado com acurácia: {min_acuracia}")

    melhor_run_metrics =  {k: (v[-1] if isinstance(v, list) else v) for k, v in melhor_run.get_metrics().items()}
    print(f"""Melhor execução:
    {melhor_run}

    Métricas:
    {melhor_run_metrics}
    Acurácia: {melhor_run_metrics.get('acuracia', -1)}""")

    python_version_utilizar=melhor_run_metrics.get('python', '3.11.11').split(' ')[0]
    sklearn_versao_utilizar = melhor_run.get_metrics().get('Versao sklearn', '1.5.0')
    print(f'python_version_utilizar: {python_version_utilizar}')
    print(f'sklearn_versao_utilizar: {sklearn_versao_utilizar}')
    try:
      datasets = [ ('baseline', Dataset.get_by_name(workspace=workspace, name='Risco-de-Credito', version= 'latest') ), ]
    except:
      datasets = []

    melhor_run.download_files("./modelo")

    model_path_padrao = 'model'
    model_path = melhor_run_metrics.get("model_binary", model_path_padrao)
    if model_path != model_path_padrao and os.path.exists(model_path):
      try:
        shutil.move(model_path, model_path_padrao)
      except:
        pass
      model_path = model_path_padrao

    modelo = melhor_run.register_model(model_name="ModeloRegistrado",
                                      model_framework=Model.Framework.SCIKITLEARN,
                                      model_framework_version=sklearn_versao_utilizar,
                                      model_path = model_path_padrao,
                                      datasets = datasets,
                                      properties = melhor_run.get_metrics(),
                                      description="Modelo criado no trabalho de mlops. \n Objetivo: Experimentos e versionamento de modelos. \n Plataforma: Azure"
                                      )
promote_models()

UserErrorException: UserErrorException:
	Message: You are currently logged-in to 11dbbfe2-89b8-4549-be10-cec364e59551 tenant. You don't have access to subscription_id subscription, please check if it is in this tenant. All the subscriptions that you have access to in this tenant are = 
 [SubscriptionInfo(subscription_name='Azure for Students', subscription_id='29e32811-9313-4d42-80fe-98720f16c8f7')]. 
 Please refer to aka.ms/aml-notebook-auth for different authentication mechanisms in azureml-sdk.
	InnerException None
	ErrorResponse 
{
    "error": {
        "code": "UserError",
        "message": "You are currently logged-in to 11dbbfe2-89b8-4549-be10-cec364e59551 tenant. You don't have access to subscription_id subscription, please check if it is in this tenant. All the subscriptions that you have access to in this tenant are = \n [SubscriptionInfo(subscription_name='Azure for Students', subscription_id='29e32811-9313-4d42-80fe-98720f16c8f7')]. \n Please refer to aka.ms/aml-notebook-